# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [9]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [10]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['year'] = pd.to_datetime(dim_tiempo['Release Date']).dt.year
dim_tiempo['month'] = pd.to_datetime(dim_tiempo['Release Date']).dt.month
dim_tiempo['day'] = pd.to_datetime(dim_tiempo['Release Date']).dt.day

dim_tiempo['id_Release_Date'] = dim_tiempo.index + 1

display(dim_tiempo)
  

,Release Date,year,month,day,id_Release_Date
0,4/26/2024,2024,4,26,1
1,5/4/2024,2024,5,4,2
2,3/19/2024,2024,3,19,3
3,1/12/2023,2023,1,12,4
4,5/31/2024,2024,5,31,5
...,...,...,...,...,...
1557,4/9/2016,2016,4,9,1558
1558,6/5/2023,2023,6,5,1559
1559,10/31/2018,2018,10,31,1560
1560,11/8/2016,2016,11,8,1561


In [11]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_Track'] = dim_pista.index + 1
display(dim_pista)

,Track,Artist,Explicit Track,id_Track
0,MILLION DOLLAR BABY,Tommy Richman,0,1
1,Not Like Us,Kendrick Lamar,1,2
2,i like the way you kiss me,Artemas,0,3
3,Flowers,Miley Cyrus,0,4
4,Houdini,Eminem,1,5
...,...,...,...,...
4531,For the Last Time,$uicideboy$,1,4532
4532,Dil Meri Na Sune,Atif Aslam,0,4533
4533,Grace (feat. 42 Dugg),Lil Baby,1,4534
4534,Nashe Si Chadh Gayi,Arijit Singh,0,4535


In [16]:
# 3. Crear fact_streams
df_temp = spotify_raw.merge(dim_tiempo, on='Release Date', how='left')
df_temp = df_temp.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')
fact_streams = df_temp[[
    'id_Release_Date',
    'id_Track',
    'Spotify Streams',
    'YouTube Views',
    'Track Score'
]]

display(fact_streams)

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score
0,1,1,"390,470,936","84,274,754",725.4
1,2,2,"323,703,884","116,347,040",545.9
2,3,3,"601,309,283","122,599,116",538.4
3,4,4,"2,031,280,633","1,096,100,899",444.9
4,5,5,"107,034,922","77,373,957",423.3
...,...,...,...,...,...
4595,1497,4532,"305,049,963","149,247,747",19.4
4596,1030,4533,"52,282,360","943,920,245",19.4
4597,746,4534,"189,972,685","201,027,333",19.4
4598,1561,4535,"145,467,020","1,118,595,159",19.4


In [17]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_Release_Date', how='left').merge(dim_pista, on='id_Track', how='left')
display(cubo_olap)

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track
0,1,1,"390,470,936","84,274,754",725.4,4/26/2024,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,"323,703,884","116,347,040",545.9,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,"601,309,283","122,599,116",538.4,3/19/2024,2024,3,19,i like the way you kiss me,Artemas,0
3,4,4,"2,031,280,633","1,096,100,899",444.9,1/12/2023,2023,1,12,Flowers,Miley Cyrus,0
4,5,5,"107,034,922","77,373,957",423.3,5/31/2024,2024,5,31,Houdini,Eminem,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4595,1497,4532,"305,049,963","149,247,747",19.4,9/5/2017,2017,9,5,For the Last Time,$uicideboy$,1
4596,1030,4533,"52,282,360","943,920,245",19.4,7/27/2018,2018,7,27,Dil Meri Na Sune,Atif Aslam,0
4597,746,4534,"189,972,685","201,027,333",19.4,2/28/2020,2020,2,28,Grace (feat. 42 Dugg),Lil Baby,1
4598,1561,4535,"145,467,020","1,118,595,159",19.4,11/8/2016,2016,11,8,Nashe Si Chadh Gayi,Arijit Singh,0


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Porque en este caso en realidad solo nos importan 2 factores, la información de la canción y la fecha. Con esa información se describe todo lo importante para la tabla de hechos

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [18]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Roll-up]
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum()
display(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: [Slice]
respuesta_2 = cubo_olap[cubo_olap['year'] == 2024]
display(respuesta_2)

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: [Drill-down]
respuesta_3 = cubo_olap.groupby(['year', 'month'])['Spotify Streams'].sum()
display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: [Dice]
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 'Yes') & (cubo_olap['year'] == 2024)]
display(respuesta_4)

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: [Roll-up]
respuesta_5 = cubo_olap.groupby('year')['Track Score'].mean()
display(respuesta_5)

Artist
"XY"                                                                                           0
$OHO BANI                                                                             54,065,563
$uicideboy$                                    44,027,794675,082,623189,588,27073,979,851204,...
&ME                                                                                   34,601,626
(G)I-DLE                                             289,304,213286,016,224242,708,36858,909,647
                                                                     ...                        
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý                                                        2,409,532
ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý)                                                         548,408
ýýýýýýýýýýýýýýýýýý                                                                     4,943,057
ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý                                              456,034
ýýýýýýýýýýýýýýýýýýýýýýý

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track
0,1,1,"390,470,936","84,274,754",725.4,4/26/2024,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,"323,703,884","116,347,040",545.9,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,"601,309,283","122,599,116",538.4,3/19/2024,2024,3,19,i like the way you kiss me,Artemas,0
4,5,5,"107,034,922","77,373,957",423.3,5/31/2024,2024,5,31,Houdini,Eminem,1
6,7,7,"900,158,751","308,723,145",407.2,1/18/2024,2024,1,18,Beautiful Things,Benson Boone,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4544,361,4481,"6,184,259","1,090,122",19.6,5/16/2024,2024,5,16,ATM,Nemzzz,1
4552,13,4489,"3,970,523","408,201",19.6,5/17/2024,2024,5,17,AirBnb,ian,1
4561,190,4498,"1,633,647","9,700,826",19.5,5/24/2024,2024,5,24,"Tu Juliet Jatt Di ( From ""Jatt & Juliet 3"")",Diljit Dosanjh,0
4593,616,4530,"3,754,914","34,315,313",19.4,3/27/2024,2024,3,27,"Jaragandi (From ""Game Changer"") (Telugu)",Thaman S,0


year  month
1987  7                                            1,879,386,440
1991  9                                            2,021,910,364
1994  1                                            1,810,649,765
1998  4                                              178,339,925
1999  6                                            1,405,353,677
                                     ...                        
2024  2        675,079,153426,700,979466,887,055251,153,86934...
      3        601,309,283355,186,436591,068,364532,012,79019...
      4        390,470,936395,433,400547,882,8711,003,215,671...
      5        323,703,884107,034,92290,676,573250,080,413221...
      6        1,653,018,119111,155,12959,314,441126,247,8861...
Name: Spotify Streams, Length: 191, dtype: object

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track


year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [25]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Slice]
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
display(respuesta_6.head(5))

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: [Drill-down]
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
display(respuesta_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: [Dice]
respuesta_8 = cubo_olap[(cubo_olap['month'] == 12) & (cubo_olap['year'] == 2023)]
display(respuesta_8)

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: [Roll-up]
respuesta_9 = cubo_olap.groupby('month')['Spotify Streams'].sum().sort_values(ascending=False)
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: [Dice]
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift') & (cubo_olap['Explicit Track'] == 0) & (cubo_olap['year'] == 2024)]
display(respuesta_10.head(5))

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track
63,57,61,"1,811,990,630","1,516,863,120",184.8,5/6/2022,2022,5,6,Me Porto Bonito,Bad Bunny,1
88,57,86,"1,514,239,759","1,366,181,141",163.1,5/6/2022,2022,5,6,Titï¿½ï¿½ Me Pregu,Bad Bunny,1
102,91,100,"478,150,858","113,379,734",150.7,10/13/2023,2023,10,13,MONACO,Bad Bunny,1
174,91,171,"689,181,455","183,381,482",114.2,10/13/2023,2023,10,13,PERRO NEGRO,Bad Bunny,1
187,57,184,"1,421,763,497","945,857,087",110.7,5/6/2022,2022,5,6,Ojitos Lindos,Bad Bunny,0


Artist                                       Track                                                              
"XY"                                         Come and Say - Original Mix                                            415,159,148
$OHO BANI                                    ZEIT, DASS SICH WAS DREHT                                                6,744,201
$uicideboy$                                  ...And to Those I Love, Thanks for Sticking Around                     269,460,083
                                             1000 Blunts                                                             87,680,985
                                             Are You Going to See the Rose in the Vase, or the Dust on the Table      2,640,169
                                                                                                                       ...     
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý              ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý                              44

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track
61,55,59,"1,384","222,076",186.5,12/31/2023,2023,12,31,Very Cute Melody by Marimba Tone (39813),mitsu sound,0
94,84,92,"649,404,552","433,839,212",155.9,12/2/2023,2023,12,2,La Diabla,Xavi,0
97,87,95,"359,803,580","369,321,334",153.1,12/7/2023,2023,12,7,BELLAKEO,Peso Pluma,1
131,112,129,"8,327",NaN,130.7,12/10/2023,2023,12,10,Looking Like a Wow End of Year 2023,WasteLand,0
177,152,174,"606,124,553","166,363,445",112.6,12/1/2023,2023,12,1,LUNA,Feid,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4189,979,4133,"100,634,965","19,367,095",20.7,12/22/2023,2023,12,22,Tacata (Remix),Tiagz,1
4194,87,4138,"9,129,729","78,290,967",20.7,12/7/2023,2023,12,7,YA NO TE EXTRAï¿½,NATTI NATASHA,0
4385,152,4325,"45,238,166","8,478,416",20.1,12/1/2023,2023,12,1,MY HOUSE,Beyoncï¿,1
4462,1533,4401,"3,225,791","72,510,719",19.8,12/29/2023,2023,12,29,ýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý,0


month
12    720,822,8681,337,619,9171,3841,873,746,537649,...
2     675,079,153426,700,979304,929,492104,271584,42...
11    670,665,438493,627,0714,281,468,7203,427,498,8...
3     601,309,283355,186,436925,655,5693,301,814,535...
4     390,470,936395,433,400547,882,8711,003,215,671...
5     323,703,884107,034,92290,676,573250,080,413221...
7     3,107,100,349670,376,179891,720,2451,692,951,4...
10    297,1851,720,918,0651,563,224,246478,150,8583,...
1     2,031,280,633900,158,7511,747,093,4053,909,458...
6     1,653,018,119111,155,129775,135,157759,998,727...
9     1,258,569,694750,154,960997,702,3041,556,275,7...
8     1,065,399,3311,658,229,4221,391,688,322609,699...
Name: Spotify Streams, dtype: object

,id_Release_Date,id_Track,Spotify Streams,YouTube Views,Track Score,Release Date,year,month,day,Track,Artist,Explicit Track
16,17,17,"395,433,400","104,673,836",297.6,4/18/2024,2024,4,18,Fortnight (feat. Post Malone),Taylor Swift,0
866,17,861,"166,168,092","16,423,009",52.3,4/18/2024,2024,4,18,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0
1013,17,1007,"175,613,480","15,237,337",47.8,4/18/2024,2024,4,18,"So Long, London",Taylor Swift,0
1167,17,1159,"161,916,445","13,509,432",44.0,4/18/2024,2024,4,18,My Boy Only Breaks His Favorite Toys,Taylor Swift,0
1270,17,1260,"165,611,453","9,873,283",42.1,4/18/2024,2024,4,18,Guilty as Sin?,Taylor Swift,0
